# SWISS Migration — Span-Wire Signal Support in CivilPy

ODOT's Office of Structural Engineering has sized **span-wire traffic-signal
supports** with **SWISS** ("Span Wire Signal Support") since the mainframe era —
most recently as a 2011 Delphi build whose source code is lost.  SWISS iterates
wire tension until the system sag matches the designer's requirement, then
reports stringing tensions, attachment heights, and pole base moments checked
against Standard Construction Drawing TC-81.10.

Two reasons to replace it:

1. **Supportability** — closed-source 32-bit binary, no source, ASD-era.
2. **Methodology** — SWISS folds wind into a flat 42-psf "design factor"
   (allowable-stress practice); new design must follow **AASHTO LRFD-LTS**.

This notebook demonstrates the replacement stack in CivilPy:

| module | replaces |
|---|---|
| `civilpy.structural.spanwire` | SWISS's sag-tension engine + `CodeList.xml` catalog |
| `civilpy.structural.aashto.lts` | the 42-psf design factor, per LRFD-LTS Art. 3.8 / Section 5 / Section 11 |

The same tools ship on the command line: `civilpy spanwire catalog | simple |
wye | system`.

## 1. The ODOT hardware catalog

SWISS shipped its signal/sign/wire tables as `CodeList.xml`; CivilPy bundles
that exact file (the 2010 backplate-prototype revision) and parses it into
typed records.

In [1]:
import pandas as pd
from civilpy.structural.spanwire import load_codelist

catalog = load_codelist()
print(f"{len(catalog.signals)} signal heads, {len(catalog.signs)} sign types, "
      f"{len(catalog.wires)} wires")
pd.DataFrame([vars(s) for s in catalog.signals.values()]).head(8)

36 signal heads, 3 sign types, 17 wires


,code,category,sections,lens_size_in,material,weight_lb,height_ft,area_sqft
0,1AA,S1A,1,8,Aluminum,20.0,1.7,0.40
1,1AABP,S1A,1,8,Al+BACKPLATE,32.4,2.3,1.80
2,1AP,S1A,1,8,Polycarbonate,15.0,1.7,0.40
3,1APBP,S1A,1,8,PC+BACKPLATE,27.4,2.3,1.80
4,1BA,S1A,1,12,Aluminum,25.0,1.9,0.56
5,1BABP,S1A,1,12,Al+BACKPLATE,38.0,2.4,1.90
6,1BP,S1A,1,12,Polycarbonate,18.0,1.9,0.56
7,1BPBP,S1A,1,12,PC+BACKPLATE,31.0,2.4,1.90


## 2. A simple span, and how the solver thinks

A messenger wire between two strain poles carries signal heads plus its own
weight.  For a trial horizontal tension $H$, the wire drops $M(x)/H$ below the
attachment chord, where $M(x)$ is the moment of the equivalent simply-supported
beam — the classic cable–beam analogy, and exactly the model SWISS iterates.
`SimpleSpan.solve` bisects $H$ until the **system sag** (highest attachment
minus lowest wire point) hits the requirement.

Check it against a closed form first: a single 100-lb load at midspan of a
weightless 100-ft wire with 5 ft of sag needs
$H = M_{max}/D = (100\cdot100/4)/5 = 500$ lb.

In [2]:
from civilpy.structural.spanwire import SimpleSpan, SpanLoad

check = SimpleSpan(100.0, loads=[SpanLoad(50.0, 100.0)]).solve(5.0)
print(f"H = {check.horizontal_tension_lb:.1f} lb   (closed form: 500.0)")

# Now a real one: two catalog heads, ODOT's default 1 lb/ft wire allowance.
s3 = catalog.signals["3BA"]     # 3-section 12-in aluminum
s5 = catalog.signals["5CA"]     # 5-section cluster
span = SimpleSpan(
    100.0, wire_weight_plf=1.0,
    loads=[SpanLoad(30.0, s3.weight_lb, s3.area_sqft, s3.code),
           SpanLoad(60.0, s5.weight_lb, s5.area_sqft, s5.code)],
)
sol = span.solve(5.0)           # max sag = 5% of the pole spacing
att = span.attachment_elevations(sol, clearance_ft=20.5)
print(f"stringing tension H = {sol.horizontal_tension_lb:.1f} lb")
print(f"reactions {sol.start_reaction_lb:.1f} / {sol.end_reaction_lb:.1f} lb, "
      f"low point at x = {sol.low_point_x_ft:.1f} ft")
print(f"attachment elevations for 20.5-ft clearance: "
      f"{att[0]:.2f} / {att[1]:.2f} ft")

H = 500.0 lb   (closed form: 500.0)
stringing tension H = 804.0 lb
reactions 124.5 / 120.5 lb, low point at x = 60.0 ft
attachment elevations for 20.5-ft clearance: 25.50 / 25.50 ft


## 3. Legacy ASD vs LRFD-LTS — the 42-psf story

SWISS's entire wind treatment was one number: a **design factor**
$\sqrt{DL^2+(\Sigma A\,q)^2}/DL \times 1.1$ with $q = 42$ psf, floored at
1.80, multiplied onto dead-load tension and base moment.  The manual traces
42 psf to "AASHTO criteria for wind pressure on traffic signals with 90 mph
winds" — the old *Standard Specifications*, i.e. ASD.

The LRFD-LTS replacement computes pressure explicitly
($P_z = 0.00256\,K_z K_d G V^2 C_d$, Extreme I, 700-yr MRI wind) and applies
it as a factored load.  Watch what it evaluates to for a signal head at a
typical attachment height in Ohio:

In [3]:
from civilpy.structural.spanwire import swiss_design_factor, pole_base_moment
from civilpy.structural.aashto import lts

V = lts.OHIO_WIND_SPEEDS[lts.MRI_YEARS["typical"]]      # 115 mph, 700-yr
z = att[0]                                              # attachment height
pz = lts.design_wind_pressure(V, z, cd=lts.CD_TRAFFIC_SIGNAL,
                              kd=lts.directionality_factor("round"))
print(f"LRFD-LTS Extreme I signal pressure at z = {z:.1f} ft: {pz:.1f} psf")
print("legacy SWISS/ODOT value:                          42.0 psf")

W = sum(l.weight_lb for l in span.loads)
A = sum(l.area_sqft for l in span.loads)
factor = max(swiss_design_factor(W, A), 1.8)
print(f"\nlegacy design factor = {factor:.2f} -> base moment = "
      f"{pole_base_moment(sol.horizontal_tension_lb, att[0], factor):,.0f} ft-lb")
print("factored LTS wind on heads:",
      ", ".join(f"{l.label}: {pz * l.area_sqft:.0f} lb" for l in span.loads))

LRFD-LTS Extreme I signal pressure at z = 25.5 ft: 41.8 psf
legacy SWISS/ODOT value:                          42.0 psf

legacy design factor = 1.80 -> base moment = 36,904 ft-lb
factored LTS wind on heads: 3BA: 67 lb, 5CA: 96 lb


The two methods land within half a psf of each other for signals at ordinary
heights — the legacy 42 psf was good physics wearing an ASD coat.  What LRFD
changes is the *bookkeeping*: explicit $K_z$ height dependence, drag by shape,
factored combinations (Table 3.4-1), and separate service and fatigue winds —
which is what lets the same library also check poles, connections, and
anchor bolts consistently.

## 4. Multi-span: the Wye

Real intersections need more than one span.  Segments join at free-floating
**bullrings**; plan equilibrium of each ring fixes the *ratios* between
segment tensions (SWISS's "tension relations"), and vertical equilibrium
floats the ring elevations.  A symmetric wye — three 100-ft legs 120° apart,
one 100-lb head mid-leg — has the closed form $H = 1000$ lb with the ring
hanging 5 ft below the poles:

In [4]:
from civilpy.structural.spanwire import SpanWireSystem

wye = SpanWireSystem.wye(
    leg_lengths=(100.0, 100.0, 100.0),
    leg_bearings_deg=(90.0, 210.0, 330.0),
    loads={leg: [SpanLoad(50.0, 100.0)] for leg in ("P1R1", "P2R1", "P3R1")},
)
wye_sol = wye.solve(5.0)
print(f"reference tension = {wye_sol.reference_tension_lb:.1f} lb "
      f"(closed form: 1000.0)")
print(f"ring elevation = {wye_sol.ring_elevations['R1']:.2f} ft")
pd.DataFrame([vars(s) for s in wye_sol.segments])

reference tension = 1000.0 lb (closed form: 1000.0)
ring elevation = -5.00 ft


,name,start,end,tension_relation,horizontal_tension_lb,start_elevation_ft,end_elevation_ft,start_reaction_lb,end_reaction_lb,low_point_x_ft,low_point_elevation_ft
0,P1R1,P1,R1,1.0,1000.000191,0.0,-4.999999,100.0,-2.131628e-14,100.0,-4.999999
1,P2R1,P2,R1,1.0,1000.000191,0.0,-4.999999,100.0,2.131628e-14,50.0,-4.999999
2,P3R1,P3,R1,1.0,1000.000191,0.0,-4.999999,100.0,0.000000e+00,100.0,-4.999999


## 5. The Box, and SWISS's famous "rotate pole 2"

Closed shapes (Delta, Box) are statically overdetermined in plan: the ring
quadrilateral plus three tails already fix what direction the fourth tail
*must* pull.  SWISS resolved this by rotating pole 2 about its bullring until
the system balanced — reporting *"for system balance rotate pole 2 N degrees
clockwise/counterclockwise"* — and the mainframe ancestor called the
alternative "warp configuration."

`SpanWireSystem` does the same: below, tail 2 of a square box is deliberately
mis-set 10° from the balancing diagonal.

In [5]:
box = SpanWireSystem.box(
    ring_positions=((0, 0), (40, 0), (40, 40), (0, 40)),
    tail_lengths=(20, 20, 20, 20),
    tail_bearings_deg=(225, 325, 45, 135),      # tail 2 should be 315
    loads={"R1R2": [SpanLoad(20.0, s3.weight_lb, s3.area_sqft, s3.code)],
           "R3R4": [SpanLoad(20.0, s3.weight_lb, s3.area_sqft, s3.code)]},
    wire_weight_plf=0.5,
)
box_sol = box.solve(4.0)
direction = "CCW" if box_sol.balance_rotation_deg > 0 else "CW"
print(f"balance: rotate {box_sol.balance_pole} "
      f"{abs(box_sol.balance_rotation_deg):.1f} deg {direction} "
      f"(in balance: {box_sol.in_balance})")
print(f"balanced P2 position: "
      f"({box_sol.balanced_pole_position[0]:.2f}, "
      f"{box_sol.balanced_pole_position[1]:.2f})")
print(f"pole stringing tensions: "
      + ", ".join(f"{p}: {t:.0f} lb" for p, t in sorted(box_sol.pole_tensions().items())))
pd.DataFrame([vars(s) for s in box_sol.segments])

balance: rotate P2 10.0 deg CW (in balance: False)
balanced P2 position: (54.14, -14.14)
pole stringing tensions: P1: 492 lb, P2: 492 lb, P3: 492 lb, P4: 492 lb


,name,start,end,tension_relation,horizontal_tension_lb,start_elevation_ft,end_elevation_ft,start_reaction_lb,end_reaction_lb,low_point_x_ft,low_point_elevation_ft
0,P1R1,P1,R1,1.414214,492.309771,0.000000,-2.132803,57.5,-47.5,20.0,-2.132803
1,P2R2,P2,R2,1.414214,492.309771,0.000000,-2.132803,57.5,-47.5,20.0,-2.132803
2,P3R3,P3,R3,1.414214,492.309771,0.000000,-2.132803,57.5,-47.5,20.0,-2.132803
3,P4R4,P4,R4,1.414214,492.309771,0.000000,-2.132803,57.5,-47.5,20.0,-2.132803
4,R1R2,R1,R2,1.000000,348.115578,-2.132803,-2.132803,37.5,37.5,20.0,-3.999999
5,R2R3,R2,R3,1.000000,348.115578,-2.132803,-2.132803,10.0,10.0,20.0,-2.420064
6,R3R4,R3,R4,1.000000,348.115578,-2.132803,-2.132803,37.5,37.5,20.0,-3.999999
7,R4R1,R4,R1,1.000000,348.115578,-2.132803,-2.132803,10.0,10.0,20.0,-2.420064


## 6. From wire to pole: LTS member checks

The wire analysis hands each pole a stringing tension and attachment height;
the LTS module takes it from there.  As a teaser, check a candidate round pole
section in flexure under the simple-span base moment (the full pole design —
combined forces, fatigue details, anchor bolts via ACI 318 — uses the same
`CheckResult` idiom as the rest of `civilpy.structural.aashto`):

In [6]:
base_moment_kip_in = pole_base_moment(sol.horizontal_tension_lb, att[0], factor) * 12 / 1000
pole = lts.RoundTube(od=12.0, t=0.25)          # trial 12-in pole section
flex = lts.round_tube_flexural_resistance(pole, f_y=50.0, m_u=base_moment_kip_in)
print(f"Mu = {flex.demand:.0f} kip-in vs phi*Mn = {flex.factored_capacity:.0f} "
      f"kip-in ({flex.details['slenderness']}) -> "
      f"D/C = {flex.demand / flex.factored_capacity:.2f}  "
      f"{'OK' if flex.ok else 'NG'}")

Mu = 443 kip-in vs phi*Mn = 1498 kip-in (noncompact) -> D/C = 0.30  OK


## 7. The same tools from the shell

```text
civilpy spanwire catalog --kind signals --match backplate
civilpy spanwire simple 100 --sag 5 --signals "30:3BA,60:5CA" --clearance 20.5
civilpy spanwire wye 100,100,100 --bearings 90,210,330 --sag 5 \
        --signals "1:50:3BA;2:50:3BA;3:50:3BA" --clearance 20.5
civilpy spanwire system intersection.json -o results.xlsx
```

`spanwire system` reads a JSON definition (wye/delta/box/custom topology,
loads by segment, clearances) and, like every CivilPy command, `-o` writes an
Excel workbook with a provenance sheet.

## Status and roadmap

**Done:** catalog, single-span and Wye/H/Delta/Box solves with SWISS-style
balance rotation, legacy design-factor parity, LTS wind/steel/fatigue checks,
CLI.

**Next:**

1. Golden-file validation against the live SWISS executable (tension
   relations, sags, base moments per configuration).
2. Complete LTS Table 3.4-1 transcription; verify signal $C_d$ and the
   non-round $K_d$ rows against the spec book; vortex shedding.
3. Combinations (two configurations sharing a pole) and the true "warp"
   mode (hold pole 2, distort the ring quadrilateral).
4. Strain-pole selection tables (TC-81.10 successor) and a printable report.